# Phase 7 — chromosome holdout and training seed robustness

This is a **stress test** of the Phase 6 MPRA activity result. It evaluates new chromosome-held-out partitions using GC, 3-mer and spaced-6/4-mer baselines alongside **three independently seeded DNABERT-2 fine-tunes per cell line**. DNABERT-2 starts from the pinned general pretrained weights, matching the best Phase 6 arm; no Phase 6 fine-tuned checkpoint is loaded. The three-seed mean score is predetermined, not selected using test labels.

Run on Kaggle **GPU**, **Internet ON**. Attach exactly:

1. `4_enhancer_phase4_external_dataset.zip` — Phase 4 labeled MPRA data.
2. `6_enhancer_phase6_mpra_results.zip` or `enhancer_phase6_mpra_results.zip` — the small Phase 6 results ZIP. It is used to authenticate the Phase 6 dataset and provide a separate reference table. The 2.59 GB Phase 6 checkpoint ZIP is **not an input**.

Kaggle may expose these ZIPs as files or extracted members. The code accepts either form. The split is fixed by seed before model fitting, and applies to both cells. Loci cannot cross chromosomes; exact or reverse-complement duplicates spanning chromosomes are removed from training/validation when they conflict with a later partition. Validation chooses C or the better of two epochs. The chromosome test set is never used for selection. **These test scores and Phase 6 scores are from different partitions; do not subtract their AUROCs as paired estimates.**

Download **both** `/kaggle/working/enhancer_phase7_chromosome_results.zip` (small tables) and `/kaggle/working/enhancer_phase7_complete_with_weights.zip` (results plus all six new `.pt` weights, roughly 3 GB). The notebook writes all checkpoints into the complete ZIP, with member SHA-256 verification. For a stopped session, attach the saved previous Phase 7 Kaggle Output and rerun cells to reuse completed arms. Save the notebook version with outputs as well.


In [1]:
# 0 — Configuration and input normalization.
import os,sys,json,zipfile,hashlib,io,shutil,gc,random,warnings,subprocess
from pathlib import Path
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import numpy as np,pandas as pd
from itertools import combinations
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler,Normalizer
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score,average_precision_score
from scipy.stats import spearmanr
ROOT=Path('/kaggle/input');assert ROOT.is_dir(),'Attach Phase 4 and Phase 6 small ZIPs.'
WORK=Path('/kaggle/working/enhancer_phase7_work');WORK.mkdir(parents=True,exist_ok=True)
CACHE=WORK/'cache';CACHE.mkdir(exist_ok=True)
WEIGHTS=WORK/'checkpoints';WEIGHTS.mkdir(exist_ok=True)
P4_SHA='b0bf4a433f1e0af74d95b8747b9312538734f70c425c8379628cafef0c947d2a'
P4_PROTOCOL='e49fe4d57dda904bffe9fa5a47d8c8e397668cbee71f24e3b0152fd1781b8705'
P6_PROTOCOL='6f11a0b1ebbb74ea5616de383b641385a272691f8b22bc32fd25284fd984f7af'
REVISION='ec1f874253852eb3907081f57294991b4280ceb6'
CONFIG={'name':'phase7_chr_holdout_v1','seed':20260930,'test_chrom_fraction':.2,
        'val_fraction_remaining_chrom':.2,'seeds':[20260930,20260931,20260932],
        'epochs':2,'batch':8,'gradient_accumulation':2,'lr':2e-5,'max_tokens':256,
        'C_candidates':[.1,1.,10.],'bootstrap_repeats':300,
        'phase4_protocol':P4_PROTOCOL,'phase6_protocol':P6_PROTOCOL,'dnabert2_revision':REVISION}
def sha(b):return hashlib.sha256(b).hexdigest()
def fsha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
LOCK=sha(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode())
def mh(z,name):
 h=hashlib.sha256()
 with z.open(name) as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
def locate(names,anchor,needed):
 hits=[]
 for root in [ROOT,Path('/kaggle/working')]:
  hits.extend(p for p in root.rglob('*.zip') if p.is_file() and p.name in names and WORK not in p.parents)
 valid=[]
 for p in hits:
  with zipfile.ZipFile(p) as z:
   if anchor in z.namelist():valid.append(p)
 if valid:
  assert len({fsha(p) for p in valid})==1,f'Ambiguous {names}: {valid}'
  return valid[0]
 candidates=[p for p in ROOT.rglob(anchor) if p.is_file()]
 assert len(candidates)==1,f'Attach one {names}, ZIP or unpacked; found {hits}; members {candidates}'
 parent=candidates[0].parent
 for n in needed:assert (parent/n).is_file(),f'Missing unpacked {n}'
 norm=WORK/'normalized';norm.mkdir(exist_ok=True)
 target=norm/names[0]
 with zipfile.ZipFile(target,'w',allowZip64=True) as z:
  for n in needed:z.write(parent/n,n,compress_type=zipfile.ZIP_STORED)
 print('Repacked unpacked Kaggle Input:',target.name)
 return target
P4=locate(['4_enhancer_phase4_external_dataset.zip','enhancer_phase4_external_dataset.zip'],
 'phase4_external_relative_activity_all_cells.csv',
 ['phase4_external_relative_activity_all_cells.csv','phase4_audit_and_provenance.json'])
P6=locate(['6_enhancer_phase6_mpra_results.zip','enhancer_phase6_mpra_results.zip'],
 'phase6_provenance.json',
 ['phase6_provenance.json','phase6_split.csv','phase6_test_metrics.csv'])
# Import completed checkpoints and score caches from a prior Phase 7 Output, if supplied.
for target,folder in [(CACHE,'cache'),(WEIGHTS,'checkpoints')]:
 files=[p for p in ROOT.rglob('*') if p.is_file() and p.parent.name==folder
        and p.parent.parent.name=='enhancer_phase7_work']
 for name in sorted({p.name for p in files}):
  options=[p for p in files if p.name==name]
  assert len({fsha(p) for p in options})==1,f'Ambiguous prior cache {name}'
  local=target/name
  if local.exists():assert fsha(local)==fsha(options[0]),f'Conflicting active cache {name}'
  else:shutil.copy2(options[0],local)
print('Phase 7 protocol SHA256',LOCK)


Repacked unpacked Kaggle Input: 4_enhancer_phase4_external_dataset.zip
Repacked unpacked Kaggle Input: 6_enhancer_phase6_mpra_results.zip
Phase 7 protocol SHA256 b92b384f8728b946f2167fb74cf4c8a7611ea658a5da758ca966d6913d564955


In [2]:
# 1 — Verify the inputs and recover the Phase 6 reference table.
with zipfile.ZipFile(P4) as z:
 assert z.testzip() is None
 audit=json.loads(z.read('phase4_audit_and_provenance.json'))
 assert audit['protocol_sha256']==P4_PROTOCOL
 name='phase4_external_relative_activity_all_cells.csv'
 assert mh(z,name)==P4_SHA==audit['output_sha256'][name]
 d=pd.read_csv(io.BytesIO(z.read(name)),dtype={'record_id':str,'chrom':str,
  'sequence':str,'sequence_sha256':str,'locus_group_id':str})
with zipfile.ZipFile(P6) as z:
 assert z.testzip() is None
 p6report=json.loads(z.read('phase6_provenance.json'))
 assert p6report['protocol_sha256']==P6_PROTOCOL
 for name,digest in p6report['result_sha256'].items():
  # Unpacked input may expose only the split and metrics; ZIP contains every member.
  if name in z.namelist():assert mh(z,name)==digest,name
 p6split=pd.read_csv(io.BytesIO(z.read('phase6_split.csv')),dtype={'record_id':str})
 p6metrics=pd.read_csv(io.BytesIO(z.read('phase6_test_metrics.csv')))
assert len(d)==len(p6split)==30340 and d.record_id.is_unique
assert d.cell_line.value_counts().to_dict()=={'K562':17198,'HepG2':13142}
assert set(d.record_id)==set(p6split.record_id)
assert all(sha(s.encode())==h for s,h in zip(d.sequence,d.sequence_sha256))
assert d.sequence.str.fullmatch('[ACGT]{200}').all()
assert set(d.label)=={0,1}
assert p6metrics.cell_line.nunique()==2
print('VERIFIED Phase 4',len(d),'rows; Phase 6 reference metrics',len(p6metrics))


VERIFIED Phase 4 30340 rows; Phase 6 reference metrics 26


In [3]:
# 2 — Freeze chromosome partitions in both cells; quarantine cross-chromosome exact/RC collisions.
chrom=np.array(sorted(d.chrom.unique()))
g=GroupShuffleSplit(n_splits=1,test_size=CONFIG['test_chrom_fraction'],random_state=CONFIG['seed'])
remain,test=next(g.split(chrom,groups=chrom))
rest=chrom[remain]
g=GroupShuffleSplit(n_splits=1,test_size=CONFIG['val_fraction_remaining_chrom'],random_state=CONFIG['seed']+1)
train,val=next(g.split(rest,groups=rest))
test_chr=set(chrom[test]);val_chr=set(rest[val]);train_chr=set(rest[train])
assert not (test_chr&val_chr or test_chr&train_chr or val_chr&train_chr)
d['split']=np.where(d.chrom.isin(test_chr),'test',np.where(d.chrom.isin(val_chr),'validation','train'))
RC=str.maketrans('ACGT','TGCA')
def revcomp(s):return s.translate(RC)[::-1]
d['canonical']=d.sequence.map(lambda s:min(s,revcomp(s)))
rank={'train':0,'validation':1,'test':2}
max_rank=d.groupby('canonical').split.transform(lambda v:max(rank[x] for x in v))
conflict=d.split.map(rank).lt(max_rank)
quarantine=d[conflict][['record_id','cell_line','chrom','split','sequence_sha256']].copy()
d=d[~conflict].copy().reset_index(drop=True)
assert d.groupby('canonical').split.nunique().max()==1
assert d.groupby('locus_group_id').split.nunique().max()==1
assert d.groupby('chrom').split.nunique().max()==1
assert set(d.split)=={'train','validation','test'}
counts=d.groupby(['cell_line','split','label']).size().unstack(fill_value=0)
assert (counts>100).all().all(),counts
print('Chromosomes: train',sorted(train_chr),'validation',sorted(val_chr),'test',sorted(test_chr))
print('Quarantined cross-partition exact/RC rows:',len(quarantine))
print(counts)


Chromosomes: train [np.str_('chr11'), np.str_('chr13'), np.str_('chr14'), np.str_('chr16'), np.str_('chr17'), np.str_('chr18'), np.str_('chr19'), np.str_('chr2'), np.str_('chr21'), np.str_('chr22'), np.str_('chr3'), np.str_('chr6'), np.str_('chr8'), np.str_('chrX'), np.str_('chrY')] validation [np.str_('chr10'), np.str_('chr15'), np.str_('chr20'), np.str_('chr5')] test [np.str_('chr1'), np.str_('chr12'), np.str_('chr4'), np.str_('chr7'), np.str_('chr9')]
Quarantined cross-partition exact/RC rows: 0
label                    0     1
cell_line split                 
HepG2     test        1764  1814
          train       3567  3777
          validation  1128  1092
K562      test        2892  2330
          train       4821  4488
          validation  1382  1285


In [4]:
# 3 — Fit strong conventional baselines using training rows only.
SCORES=[];CHOICES=[]
def emit(cell,arm,frame,score):
 values=np.asarray(score,dtype=float)
 assert len(values)==len(frame) and np.isfinite(values).all() and np.all((0<=values)&(values<=1))
 SCORES.extend({'record_id':rid,'cell_line':cell,'model':arm,'split':sp,'label':int(y),
                'locus_group_id':locus,'score':float(v)}
  for rid,sp,y,locus,v in zip(frame.record_id,frame.split,frame.label,frame.locus_group_id,values))
def gc_features(seq):return np.array([[(s.count('G')+s.count('C'))/200,
 s.count('CG')/199] for s in seq],dtype=np.float32)
def token3(s):return [s[i:i+3] for i in range(198)]
MASKS=tuple(combinations(range(6),4))
def spaced(s):return [str(mi)+''.join(s[i+j] for j in mask)
 for i in range(195) for mi,mask in enumerate(MASKS)]
def fit_classical(cell,arm):
 sub=d[d.cell_line==cell];tr=sub[sub.split=='train'];va=sub[sub.split=='validation'];te=sub[sub.split=='test']
 cache=CACHE/f'{cell}_{arm}_scores.csv';meta=CACHE/f'{cell}_{arm}_meta.json'
 expected=pd.concat([va,te])
 if cache.exists() and meta.exists():
  m=json.loads(meta.read_text());assert m['lock']==LOCK and fsha(cache)==m['sha256']
  old=pd.read_csv(cache,dtype={'record_id':str,'split':str,'score':float})
  assert old.record_id.tolist()==expected.record_id.tolist() and old.split.tolist()==expected.split.tolist()
  CHOICES.append(m['choice']);emit(cell,arm,expected,old.score)
  print('REUSED',cell,arm);return
 seq=tr.sequence.tolist()+list(map(revcomp,tr.sequence.tolist()))
 y=np.tile(tr.label.to_numpy(dtype=int),2)
 if arm=='gc_logreg':
  X=gc_features(seq);V=gc_features(va.sequence);T=gc_features(te.sequence)
  def train(c):return make_pipeline(StandardScaler(),LogisticRegression(C=c,max_iter=1200)).fit(X,y)
 elif arm=='k3_logreg':
  vec=CountVectorizer(analyzer=token3,lowercase=False,dtype=np.float32)
  X=vec.fit_transform(seq);V=vec.transform(va.sequence);T=vec.transform(te.sequence)
  def train(c):return make_pipeline(Normalizer(norm='l1'),LogisticRegression(C=c,max_iter=1200,solver='liblinear')).fit(X,y)
 elif arm=='spaced6_4_logreg':
  vec=TfidfVectorizer(analyzer=spaced,lowercase=False,dtype=np.float32,norm='l2')
  X=vec.fit_transform(seq);V=vec.transform(va.sequence);T=vec.transform(te.sequence)
  def train(c):return LogisticRegression(C=c,max_iter=1200,solver='liblinear').fit(X,y)
 else:raise ValueError(arm)
 candidates=[]
 with warnings.catch_warnings():
  warnings.simplefilter('ignore')
  for c in CONFIG['C_candidates']:
   model=train(c);pv=model.predict_proba(V)[:,1]
   candidates.append((roc_auc_score(va.label,pv),c,model,pv))
 best=max(candidates,key=lambda row:(row[0],-row[1]))
 choice={'cell_line':cell,'model':arm,'selected_C':float(best[1]),'selected_epoch':'',
  'validation_auroc':float(best[0])}
 CHOICES.append(choice)
 tv=best[2].predict_proba(T)[:,1]
 emit(cell,arm,va,best[3]);emit(cell,arm,te,tv)
 temp=cache.with_suffix('.part')
 pd.DataFrame({'record_id':expected.record_id,'split':expected.split,
  'score':np.r_[best[3],tv]}).to_csv(temp,index=False,float_format='%.12g')
 os.replace(temp,cache)
 meta.write_text(json.dumps({'lock':LOCK,'sha256':fsha(cache),'choice':choice},sort_keys=True))
 print(cell,arm,'validation AUROC',round(best[0],4),'C',best[1],flush=True)
 del seq,X,V,T,candidates
 gc.collect()
for cell in ('HepG2','K562'):
 for arm in ('gc_logreg','k3_logreg','spaced6_4_logreg'):fit_classical(cell,arm)


HepG2 gc_logreg validation AUROC 0.6443 C 0.1
HepG2 k3_logreg validation AUROC 0.7451 C 10.0
HepG2 spaced6_4_logreg validation AUROC 0.8896 C 10.0
K562 gc_logreg validation AUROC 0.6157 C 1.0
K562 k3_logreg validation AUROC 0.6579 C 10.0
K562 spaced6_4_logreg validation AUROC 0.8595 C 10.0


In [5]:
# 4 — Pretrained DNABERT-2: three independently seeded fine-tunes per cell.
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
assert torch.cuda.is_available(),'Choose a Kaggle GPU.'
DEVICE=torch.device('cuda:0')
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
try:torch.use_deterministic_algorithms(True,warn_only=True)
except Exception:pass
from importlib.metadata import version,PackageNotFoundError
try:installed=version('transformers')
except PackageNotFoundError:installed='missing'
if installed!='4.44.2':
 assert 'transformers' not in sys.modules,'Restart the session: Transformers imported before pinning.'
 subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.44.2'],check=True,timeout=900)
from transformers import AutoTokenizer,AutoModel
import transformers
assert transformers.__version__=='4.44.2'
def encoder():
 remote='zhihan1996/DNABERT-2-117M'
 tok=AutoTokenizer.from_pretrained(remote,revision=REVISION,trust_remote_code=True)
 try:enc=AutoModel.from_pretrained(remote,revision=REVISION,trust_remote_code=True)
 except (ValueError,TypeError):
  from transformers.models.bert.configuration_bert import BertConfig
  cfg=BertConfig.from_pretrained(remote,revision=REVISION)
  enc=AutoModel.from_pretrained(remote,revision=REVISION,trust_remote_code=True,config=cfg)
 import importlib
 mod=importlib.import_module(type(enc).__module__)
 assert hasattr(mod,'BertUnpadSelfAttention') and hasattr(mod,'flash_attn_qkvpacked_func')
 mod.flash_attn_qkvpacked_func=None
 return tok,enc
def tokenize(tok,seq):
 b=tok(list(seq),padding=True,add_special_tokens=True,truncation=False,return_tensors='pt')
 assert b['input_ids'].shape[1]<=CONFIG['max_tokens'],b['input_ids'].shape
 return {k:v.to(DEVICE) for k,v in b.items() if k in ('input_ids','attention_mask')}
class Net(nn.Module):
 def __init__(self,enc):
  super().__init__();self.encoder=enc
  self.dropout=nn.Dropout(.1);self.head=nn.Linear(int(enc.config.hidden_size),2)
 def forward(self,b):
  out=self.encoder(**b);h=getattr(out,'last_hidden_state',None)
  if h is None:h=out[0]
  mask=b['attention_mask'].to(h.dtype).unsqueeze(-1)
  vec=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
  return self.head(self.dropout(vec))
def predict(net,tok,seq):
 net.eval();out=[]
 with torch.inference_mode():
  for i in range(0,len(seq),32):
   b=tokenize(tok,seq[i:i+32])
   with torch.autocast('cuda',dtype=torch.float16):
    out.extend(torch.softmax(net(b).float(),dim=1)[:,1].cpu().numpy())
 return np.array(out)
def run_seed(cell,seed):
 arm=f'dnabert2_seed_{seed}'
 sub=d[d.cell_line==cell];tr=sub[sub.split=='train'];va=sub[sub.split=='validation'];te=sub[sub.split=='test']
 expected=pd.concat([va,te])
 ckpt=WEIGHTS/f'{cell}_{arm}_best.pt'
 cache=CACHE/f'{cell}_{arm}_scores.csv';meta=CACHE/f'{cell}_{arm}_complete.json'
 if ckpt.exists() and cache.exists() and meta.exists():
  m=json.loads(meta.read_text())
  assert m['lock']==LOCK and fsha(ckpt)==m['checkpoint_sha256'] and fsha(cache)==m['scores_sha256']
  old=pd.read_csv(cache,dtype={'record_id':str,'split':str,'score':float})
  assert old.record_id.tolist()==expected.record_id.tolist() and old.split.tolist()==expected.split.tolist()
  CHOICES.append(m['choice']);emit(cell,arm,expected,old.score)
  print('REUSED complete',cell,seed,flush=True);return
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
 tok,enc=encoder();net=Net(enc).to(DEVICE)
 class AugRows(Dataset):
  def __len__(self):return 2*len(tr)
  def __getitem__(self,i):
   r=tr.iloc[i%len(tr)]
   return (revcomp(r.sequence) if i>=len(tr) else r.sequence),int(r.label)
 def collate(items):return tokenize(tok,[s for s,_ in items]),torch.tensor([y for _,y in items],dtype=torch.long,device=DEVICE)
 loader=DataLoader(AugRows(),batch_size=CONFIG['batch'],shuffle=True,num_workers=0,
  collate_fn=collate,generator=torch.Generator().manual_seed(seed))
 opt=torch.optim.AdamW(net.parameters(),lr=CONFIG['lr'],weight_decay=.01)
 scaler=torch.amp.GradScaler('cuda',enabled=True)
 best=-1.;epoch=0;best_val=None
 for ep in range(1,CONFIG['epochs']+1):
  net.train();opt.zero_grad(set_to_none=True);losses=[]
  for step,(b,y) in enumerate(loader):
   with torch.autocast('cuda',dtype=torch.float16):loss=nn.functional.cross_entropy(net(b),y)
   assert torch.isfinite(loss),(cell,seed,ep,step)
   scaler.scale(loss/CONFIG['gradient_accumulation']).backward()
   if (step+1)%CONFIG['gradient_accumulation']==0 or step+1==len(loader):
    scaler.unscale_(opt);nn.utils.clip_grad_norm_(net.parameters(),1.)
    scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
   losses.append(float(loss.detach().cpu()))
  pv=predict(net,tok,va.sequence.tolist());auc=roc_auc_score(va.label,pv)
  print(cell,seed,'epoch',ep,'loss',round(float(np.mean(losses)),4),
   'validation AUROC',round(auc,4),flush=True)
  if auc>best:
   best=auc;epoch=ep;best_val=pv
   part=ckpt.with_suffix('.part')
   torch.save({'lock':LOCK,'revision':REVISION,'cell_line':cell,'seed':seed,
    'epoch':ep,'validation_auroc':float(auc),
    'state_dict':{k:v.detach().cpu() for k,v in net.state_dict().items()}},part)
   os.replace(part,ckpt)
 del loader,opt,scaler,net,enc
 gc.collect();torch.cuda.empty_cache()
 tok,enc=encoder();net=Net(enc).to(DEVICE)
 saved=torch.load(ckpt,map_location='cpu',weights_only=True)
 assert saved['lock']==LOCK and saved['revision']==REVISION
 net.load_state_dict(saved['state_dict'],strict=True)
 tp=predict(net,tok,te.sequence.tolist())
 choice={'cell_line':cell,'model':arm,'selected_C':'','selected_epoch':int(epoch),
  'validation_auroc':float(best)}
 CHOICES.append(choice)
 emit(cell,arm,va,best_val);emit(cell,arm,te,tp)
 temp=cache.with_suffix('.part')
 pd.DataFrame({'record_id':expected.record_id,'split':expected.split,
  'score':np.r_[best_val,tp]}).to_csv(temp,index=False,float_format='%.12g')
 os.replace(temp,cache)
 meta.write_text(json.dumps({'lock':LOCK,'checkpoint_sha256':fsha(ckpt),
  'scores_sha256':fsha(cache),'choice':choice},sort_keys=True))
 del net,enc,tok,saved
 gc.collect();torch.cuda.empty_cache()
for cell in ('HepG2','K562'):
 for seed in CONFIG['seeds']:run_seed(cell,seed)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 439.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.0 MB/s eta 0:00:00


tokenizer_config.json:   0%|          | 0.00/158 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/904 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


K562 20260931 epoch 1 loss 0.5025 validation AUROC 0.8736
K562 20260931 epoch 2 loss 0.3432 validation AUROC 0.8821


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


K562 20260932 epoch 1 loss 0.4921 validation AUROC 0.8866
K562 20260932 epoch 2 loss 0.3385 validation AUROC 0.8923


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
# 5 — Three-seed ensemble, chromosome test metrics, GC and chromosome strata.
scores = pd.DataFrame(SCORES)

assert not scores.duplicated(['record_id', 'cell_line', 'model']).any()
needed = {'gc_logreg', 'k3_logreg', 'spaced6_4_logreg'} | {
    f'dnabert2_seed_{seed}' for seed in CONFIG['seeds']
}
assert set(scores.model) == needed

for cell in ('HepG2', 'K562'):
    sub = scores[scores.cell_line == cell]
    expected = set(
        d[(d.cell_line == cell) & (d.split != 'train')].record_id
    )
    for arm in needed:
        assert set(sub[sub.model == arm].record_id) == expected, (cell, arm)

# Build the ensemble once, after checking both cell lines.
assert d.record_id.is_unique
assert not scores.duplicated(['record_id', 'model']).any()

wide = scores.pivot(index='record_id', columns='model', values='score')
seed_models = [f'dnabert2_seed_{seed}' for seed in CONFIG['seeds']]
assert not wide[seed_models].isna().any().any()

ens = wide[seed_models].mean(axis=1)
base = d[d.split != 'train'].set_index('record_id').loc[ens.index]

extra = pd.DataFrame({
    'record_id': ens.index,
    'score': ens.values,
    'model': 'dnabert2_three_seed_mean',
    'cell_line': base.cell_line.values,
    'split': base.split.values,
    'label': base.label.values,
    'locus_group_id': base.locus_group_id.values,
})
scores = pd.concat([scores, extra], ignore_index=True)
assert not scores.duplicated(['record_id', 'model']).any()

truth = d[['record_id', 'chrom', 'mean_log2_activity', 'sequence']]
scores = scores.merge(truth, on='record_id', validate='many_to_one')
scores['gc_count'] = scores.sequence.map(
    lambda s: s.count('G') + s.count('C')
)

def gc_pair_auc(y, s, g):
    wins = 0.0
    pairs = 0
    for count in np.unique(g):
        pos = np.sort(s[(y == 1) & (g == count)])
        neg = np.sort(s[(y == 0) & (g == count)])
        if not len(pos) or not len(neg):
            continue
        for value in pos:
            lo = np.searchsorted(neg, value, 'left')
            hi = np.searchsorted(neg, value, 'right')
            wins += lo + 0.5 * (hi - lo)
        pairs += len(pos) * len(neg)
    return (wins / pairs if pairs else np.nan), pairs

def measure(part):
    y = part.label.to_numpy(dtype=int)
    s = part.score.to_numpy(dtype=float)
    gc_auc, pairs = gc_pair_auc(
        y, s, part.gc_count.to_numpy(dtype=int)
    )
    return {
        'n': len(y),
        'positive': int(y.sum()),
        'auroc': roc_auc_score(y, s),
        'auprc': average_precision_score(y, s),
        'spearman_activity': spearmanr(
            part.mean_log2_activity.to_numpy(dtype=float), s
        ).statistic,
        'exact_gc_pair_auc': gc_auc,
        'exact_gc_pairs': pairs,
    }

test = scores[scores.split == 'test'].copy()
metrics = pd.DataFrame([
    {'cell_line': cell, 'model': model, **measure(part)}
    for (cell, model), part in test.groupby(['cell_line', 'model'])
])

print(
    metrics.sort_values(
        ['cell_line', 'auroc'], ascending=[True, False]
    )[
        ['cell_line', 'model', 'n', 'auroc', 'auprc',
         'exact_gc_pair_auc']
    ].to_string(index=False)
)

strata = []
for (cell, model), part in test.groupby(['cell_line', 'model']):
    for chrom, group in part.groupby('chrom'):
        if len(group) >= 30 and group.label.nunique() == 2:
            strata.append({
                'cell_line': cell,
                'model': model,
                'stratum': 'chromosome',
                'value': chrom,
                **measure(group),
            })

    gc_bins = pd.qcut(part.gc_count, 5, duplicates='drop')
    for interval, group in part.groupby(gc_bins, observed=True):
        if len(group) >= 30 and group.label.nunique() == 2:
            strata.append({
                'cell_line': cell,
                'model': model,
                'stratum': 'GC quintile',
                'value': str(interval),
                **measure(group),
            })

strata = pd.DataFrame(strata)

cell_line                    model    n    auroc    auprc  exact_gc_pair_auc
    HepG2 dnabert2_three_seed_mean 3578 0.910145 0.910285           0.888722
    HepG2   dnabert2_seed_20260930 3578 0.907552 0.907802           0.884676
    HepG2   dnabert2_seed_20260931 3578 0.905079 0.899107           0.882760
    HepG2   dnabert2_seed_20260932 3578 0.900578 0.899881           0.876557
    HepG2         spaced6_4_logreg 3578 0.889216 0.891182           0.869191
    HepG2                k3_logreg 3578 0.739101 0.738644           0.691944
    HepG2                gc_logreg 3578 0.648971 0.671274           0.515342
     K562 dnabert2_three_seed_mean 5222 0.908261 0.900620           0.885707
     K562   dnabert2_seed_20260932 5222 0.904971 0.893874           0.881463
     K562   dnabert2_seed_20260930 5222 0.901909 0.893745           0.880348
     K562   dnabert2_seed_20260931 5222 0.895715 0.886497           0.870756
     K562         spaced6_4_logreg 5222 0.880985 0.858625           0.859972

In [8]:
# 6 — Paired chromosome/locus bootstrap for the prespecified best-versus-baseline contrast.
boot=[]
for ci,cell in enumerate(('HepG2','K562')):
 sub=test[test.cell_line==cell]
 wide=sub.pivot(index='record_id',columns='model',values='score')
 base=d.set_index('record_id').loc[wide.index]
 assert not wide.isna().any().any()
 labels=base.label.to_numpy(dtype=int)
 loci=base.locus_group_id.to_numpy();u=np.unique(loci)
 by=[np.flatnonzero(loci==g) for g in u]
 rng=np.random.default_rng(CONFIG['seed']+751+ci)
 for repeat in range(CONFIG['bootstrap_repeats']):
  draw=rng.integers(0,len(u),size=len(u))
  idx=np.concatenate([by[i] for i in draw])
  if np.unique(labels[idx]).size<2:continue
  auc={m:roc_auc_score(labels[idx],wide[m].to_numpy()[idx])
       for m in ['dnabert2_three_seed_mean','spaced6_4_logreg','gc_logreg']}
  boot.append({'cell_line':cell,'repeat':repeat,'dnabert2_ensemble_auroc':auc['dnabert2_three_seed_mean'],
   'spaced6_4_auroc':auc['spaced6_4_logreg'],
   'delta_vs_spaced6_4':auc['dnabert2_three_seed_mean']-auc['spaced6_4_logreg'],
   'delta_vs_gc':auc['dnabert2_three_seed_mean']-auc['gc_logreg']})
boot=pd.DataFrame(boot)
for cell,g in boot.groupby('cell_line'):
 assert len(g)>=290
 print(cell,'ensemble – spaced6_4 bootstrap 95% interval',
       np.quantile(g.delta_vs_spaced6_4,[.025,.5,.975]).round(4).tolist())
print('Phase 6 reference only, different holdout:')
print(p6metrics[p6metrics.model.isin(['dnabert2_pretrained','spaced6_4_logreg'])][
 ['cell_line','model','auroc']].to_string(index=False))


HepG2 ensemble – spaced6_4 bootstrap 95% interval [0.0129, 0.0208, 0.0304]
K562 ensemble – spaced6_4 bootstrap 95% interval [0.0197, 0.0277, 0.0355]
Phase 6 reference only, different holdout:
cell_line               model    auroc
    HepG2 dnabert2_pretrained 0.916174
    HepG2    spaced6_4_logreg 0.898582
     K562 dnabert2_pretrained 0.911758
     K562    spaced6_4_logreg 0.882250


In [9]:
# 7 — Save all metrics, split, diagnostics and weights in two verified archives.
OUT=Path('/kaggle/working/enhancer_phase7_chromosome_results.zip')
FULL=Path('/kaggle/working/enhancer_phase7_complete_with_weights.zip')
ART=WORK/'results';ART.mkdir(exist_ok=True)
files={}
def csv(name,data):
 p=ART/name;data.to_csv(p,index=False,float_format='%.12g');files[name]=p
csv('phase7_split.csv',d[['record_id','cell_line','chrom','locus_group_id','split','label']])
csv('phase7_quarantined_exact_rc.csv',quarantine)
csv('phase7_validation_choices.csv',pd.DataFrame(CHOICES))
csv('phase7_scores_validation_test.csv',scores.drop(columns='sequence'))
csv('phase7_test_metrics.csv',metrics)
csv('phase7_strata.csv',strata)
csv('phase7_locus_bootstrap.csv',boot)
csv('phase7_phase6_reference_metrics.csv',p6metrics)
weight_paths={f'{cell}_dnabert2_seed_{seed}_best.pt':WEIGHTS/f'{cell}_dnabert2_seed_{seed}_best.pt'
 for cell in ('HepG2','K562') for seed in CONFIG['seeds']}
assert len(weight_paths)==6 and all(p.is_file() for p in weight_paths.values())
report={'protocol_sha256':LOCK,'config':CONFIG,'phase4_sha256':fsha(P4),'phase6_sha256':fsha(P6),
 'split_chromosomes':{'train':sorted(train_chr),'validation':sorted(val_chr),'test':sorted(test_chr)},
 'quarantine_count':len(quarantine),'result_sha256':{n:fsha(p) for n,p in files.items()},
 'checkpoints':{n:{'sha256':fsha(p),'bytes':p.stat().st_size} for n,p in weight_paths.items()},
 'selection':'validation AUROC only; C grid for baselines; 2 epochs for each independent DNABERT-2 seed',
 'scope':'Within-design top/bottom MPRA activity; a chromosome-disjoint stress test, not a new independent assay',
 'comparison_caveat':'Phase 6 and Phase 7 test AUROCs use different genomic rows. Their difference is not paired.',
 'similarity_caveat':'Overlap within chromosome and exact/RC cross-chromosome collisions controlled; distant homologous loci may remain related.'}
rp=ART/'phase7_provenance.json';rp.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n');files[rp.name]=rp
with zipfile.ZipFile(OUT,'w',allowZip64=True) as z:
 for n,p in files.items():z.write(p,n,compress_type=zipfile.ZIP_DEFLATED)
with zipfile.ZipFile(OUT) as z:
 assert z.testzip() is None
 for n,h in report['result_sha256'].items():assert mh(z,n)==h
with zipfile.ZipFile(FULL,'w',allowZip64=True) as z:
 z.write(OUT,OUT.name,compress_type=zipfile.ZIP_STORED)
 for n,p in weight_paths.items():z.write(p,n,compress_type=zipfile.ZIP_STORED)
 z.writestr('phase7_complete_manifest.json',json.dumps({'protocol_sha256':LOCK,
  'results_zip_sha256':fsha(OUT),'checkpoints':report['checkpoints']},indent=2,sort_keys=True)+'\n')
with zipfile.ZipFile(FULL) as z:
 assert z.testzip() is None
 assert mh(z,OUT.name)==fsha(OUT)
 for n,v in report['checkpoints'].items():assert mh(z,n)==v['sha256']
print('DOWNLOAD',OUT,OUT.stat().st_size,'SHA256',fsha(OUT))
print('DOWNLOAD',FULL,FULL.stat().st_size,'SHA256',fsha(FULL))
from IPython.display import FileLink,display
os.chdir('/kaggle/working')
display(FileLink(OUT.name));display(FileLink(FULL.name))


DOWNLOAD /kaggle/working/enhancer_phase7_chromosome_results.zip 3379610 SHA256 c9d0973201ee77a066f6310824fcf7f7e08865dd2dc8467c3a58a8250749c4df
DOWNLOAD /kaggle/working/enhancer_phase7_complete_with_weights.zip 2813362280 SHA256 76c97cdf80ccedc41de0242fa92de5961bc9620005958cba828675d114231aa1


/kaggle/working/enhancer_phase7_chromosome_results.zip

/kaggle/working/enhancer_phase7_complete_with_weights.zip